# Tour API 관광지 데이터 수집 및 정제 (메인)

전국 관광지 기본 데이터를 수집하고 법정동명, 카테고리를 결합하여 시뮬레이터용 데이터셋(`관광정보_메인_장소_데이터.csv`)을 생성하는 파일입니다.

### 주요 정제 작업:
1. **행정구역 및 주소 보정**: 누락되거나 오류가 있는 지역 코드를 실제 주소(`addr1`) 텍스트 기반으로 보정.
2. **이상치 좌표 처리**: 좌표(`mapx`, `mapy`)가 없거나 대한민국 영토 범위를 벗어난 이상치 데이터는 제거하고, 위경도가 바뀐 경우 자동으로 위치를 수정.
3. **중복 명소 제거**: 명소 이름과 좌표(소수점 4자리 기준)가 겹치는 중복 데이터를 1개로 통합합니다.
4. **'여행 코스' 제외**: 단일 장소가 아닌 코스 형태의 데이터는 별도 파일(`관광정보_여행코스_참조_데이터.csv`)로 분리하여 내보냄.

## 1. 환경 설정 및 API 공통 함수 정의

In [53]:
!pip install requests pandas python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv

# .env 파일에서 API 키 로드 (상위 폴더 확인)
load_dotenv(dotenv_path="../../.env")
API_KEY = os.getenv("TOUR_API_DECODE_KEY")
BASE_URL = "https://apis.data.go.kr/B551011/KorService2"

print("API 키 로드 상태:", "성공" if API_KEY else "실패 (내부 .env 파일과 변수명을 확인하세요)")

def fetch_api_data(endpoint, base_url=BASE_URL, params=None):
    url = f"{base_url}/{endpoint}"
    default_params = {
        "serviceKey": API_KEY,
        "MobileOS": "ETC",
        "MobileApp": "RouteCheck",
        "_type": "json"
    }
    if params:
        default_params.update(params)
    
    max_retries = 3
    retry_delay = 2  # base delay in seconds
    
    for attempt in range(max_retries + 1):
        try:
            response = requests.get(url, params=default_params, timeout=10)
            
            # [HTTP 429: Too Many Requests 대응]
            if response.status_code == 429:
                if attempt < max_retries:
                    sleep_time = retry_delay * (2 ** attempt)
                    print(f"[WARNING] API 요청 제한(HTTP 429) 발생! {sleep_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
                    time.sleep(sleep_time)
                    continue
                else:
                    print(f"[ERROR] API 요청 제한(HTTP 429) 초과로 요청 최종 실패 ({endpoint})")
                    return pd.DataFrame()
            
            if response.status_code == 200:
                res_json = response.json()
                if 'response' in res_json:
                    header = res_json['response'].get('header', {})
                    result_code = header.get('resultCode')
                    if result_code != '0000':
                        # [공공데이터 포털 서버단 트래픽 제한 에러 대응 (예: '04' 또는 '22')]
                        if result_code in ['04', '22'] and attempt < max_retries:
                            sleep_time = retry_delay * (2 ** attempt)
                            print(f"[WARNING] API 트래픽 초과 에러 [{result_code}] 발생! {sleep_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
                            time.sleep(sleep_time)
                            continue
                        print(f"API 자체 에러 [{result_code}]: {header.get('resultMsg')} ({endpoint})")
                        return pd.DataFrame()
                    
                    body = res_json['response'].get('body', {})
                    items = body.get('items', {})
                    if not items or items == "":
                        return pd.DataFrame()
                    
                    if 'item' in items:
                        item_list = items['item']
                        if isinstance(item_list, dict):
                            item_list = [item_list]
                        return pd.DataFrame(item_list)
            else:
                if attempt < max_retries:
                    print(f"[WARNING] HTTP Error Code: {response.status_code} ({url}). {retry_delay}초 대기 후 재시도...")
                    time.sleep(retry_delay)
                    continue
                else:
                    print(f"HTTP Error Code: {response.status_code} ({url})")
                    return pd.DataFrame()
        except Exception as e:
            if attempt < max_retries:
                print(f"[WARNING] 네트워크 또는 데이터 파싱 실패 ({endpoint}): {e}. {retry_delay}초 대기 후 재시도...")
                time.sleep(retry_delay)
                continue
            else:
                print(f"네트워크 또는 데이터 파싱 최종 실패 ({endpoint}): {e}")
                return pd.DataFrame()
                
    return pd.DataFrame()

API 키 로드 상태: 성공


## 2. 관광지 기본 목록 수집 및 좌표 정제

In [55]:
print("\n--- 관광지 목록 수집 (areaBasedList2) ---")

all_dfs = []
page_no = 1
rows_per_page = 5000
print("전체 데이터 수집 시작...")

while True:
    print(f"페이지 {page_no} 수집 중...")
    params = {
        "numOfRows": rows_per_page,
        "pageNo": page_no
    }
    df_chunk = fetch_api_data(endpoint="areaBasedList2", params=params)

    if df_chunk.empty:
        break
    all_dfs.append(df_chunk)

    if len(df_chunk) < rows_per_page:
        break
    
    page_no += 1
    time.sleep(0.3)

df_area_list_all = pd.concat(all_dfs, ignore_index=True)
print("수집 완료! 총 데이터 개수:", df_area_list_all.shape)

# 불필요한 매칭용 외래키 areacode/sigungucode 제거
columns_to_drop = ['areacode', 'sigungucode', 'cat1', 'cat2', 'cat3']
df_area_list_all.drop(columns=columns_to_drop, inplace=True, errors='ignore')
print("기본 컬럼 제거 완료!", df_area_list_all.shape)


--- 관광지 목록 수집 (areaBasedList2) ---
전체 데이터 수집 시작...
페이지 1 수집 중...
페이지 2 수집 중...
페이지 3 수집 중...
페이지 4 수집 중...
페이지 5 수집 중...
페이지 6 수집 중...
페이지 7 수집 중...
페이지 8 수집 중...
페이지 9 수집 중...
페이지 10 수집 중...
페이지 11 수집 중...
수집 완료! 총 데이터 개수: (50735, 25)
기본 컬럼 제거 완료! (50735, 20)


In [56]:
print("\n--- 좌표(mapx, mapy) 결측치 및 이상치 제거 및 보정 ---")

df_area_list_all['mapx_num'] = pd.to_numeric(df_area_list_all['mapx'].replace("null", None).replace("", None), errors='coerce')
df_area_list_all['mapy_num'] = pd.to_numeric(df_area_list_all['mapy'].replace("null", None).replace("", None), errors='coerce')

# 1. 위경도 뒤바뀐 데이터 보정 (mapx가 위도 33~39 범위이고 mapy가 경도 124~132 범위인 경우 swap)
swapped_mask = (df_area_list_all['mapx_num'] >= 33.0) & (df_area_list_all['mapx_num'] <= 39.0) & \
               (df_area_list_all['mapy_num'] >= 124.0) & (df_area_list_all['mapy_num'] <= 132.0)

print(f"위경도 뒤바뀜 감지 및 자동 보정 대상 행 개수: {swapped_mask.sum()}개")

# 스왑 보정 수행
if swapped_mask.sum() > 0:
    temp_x = df_area_list_all.loc[swapped_mask, 'mapx_num'].copy()
    df_area_list_all.loc[swapped_mask, 'mapx_num'] = df_area_list_all.loc[swapped_mask, 'mapy_num']
    df_area_list_all.loc[swapped_mask, 'mapy_num'] = temp_x
    # 원본 문자열 컬럼도 업데이트
    df_area_list_all.loc[swapped_mask, 'mapx'] = df_area_list_all.loc[swapped_mask, 'mapx_num'].astype(str)
    df_area_list_all.loc[swapped_mask, 'mapy'] = df_area_list_all.loc[swapped_mask, 'mapy_num'].astype(str)

# 2. 결측치 및 한국 범위 외부 이상치 필터링
invalid_coords_mask = (
    df_area_list_all['mapx_num'].isna() |
    df_area_list_all['mapy_num'].isna() |
    (df_area_list_all['mapx_num'] == 0) |
    (df_area_list_all['mapy_num'] == 0) |
    (df_area_list_all['mapx_num'] < 124.0) | (df_area_list_all['mapx_num'] > 132.0) |
    (df_area_list_all['mapy_num'] < 33.0) | (df_area_list_all['mapy_num'] > 39.0)
)

print(f"좌표 정보가 누락되었거나 비정상(0, null, 한국 범위 외)인 행 개수: {invalid_coords_mask.sum()}개 (삭제 대상)")

df_area_list_all = df_area_list_all[~invalid_coords_mask].copy()
df_area_list_all.drop(columns=['mapx_num', 'mapy_num'], inplace=True, errors='ignore')

# 3. 장소명(title) HTML 태그 및 불필요 대괄호 설명 정제
import re
def clean_title(t):
    if not isinstance(t, str):
        return ""
    t = re.sub(r'<[^>]*>', '', t)
    t = re.sub(r'\[[^\]]*\]', '', t)
    return t.strip()

df_area_list_all['title'] = df_area_list_all['title'].apply(clean_title)
print(f"좌표 결측치/이상치 제거 및 장소명 태그 정제 완료! 최종 데이터 크기: {df_area_list_all.shape}")



--- 좌표(mapx, mapy) 결측치 및 이상치 제거 ---
좌표(mapx, mapy) 정보가 누락되었거나 비정상(0 또는 null)인 행 개수: 58개 (삭제 대상)
좌표 결측치 제거 완료! (주소만 누락되고 좌표는 생존한 명소는 그대로 유지) 최종 데이터 크기: (50677, 20)


## 3. 법정동 및 카테고리 매칭 & 주소 기반 지역명 보정

In [57]:
print("\n--- 법정동 코드 조회 (ldongCode2) ---")

sido_dfs = []
page_no = 1
rows_per_page = 500

while True:
    sido_params = {
        "lDongListYn": "Y",
        "numOfRows": rows_per_page,
        "pageNo": page_no
    }
    df_sido_chunk = fetch_api_data(endpoint="ldongCode2", params=sido_params)

    if df_sido_chunk.empty:
        break
    sido_dfs.append(df_sido_chunk)

    if len(df_sido_chunk) < rows_per_page:
        break
    
    page_no += 1
    time.sleep(0.2)

df_sido = pd.concat(sido_dfs, ignore_index=True)
print("법정동 코드 수집 완료! 총 개수:", df_sido.shape)
display(df_sido.head(5))


--- 법정동 코드 조회 (ldongCode2) ---
법정동 코드 수집 완료! 총 개수: (268, 5)


,lDongRegnCd,lDongRegnNm,lDongSignguCd,lDongSignguNm,rnum
0,11,서울특별시,110,종로구,1
1,11,서울특별시,140,중구,2
2,11,서울특별시,170,용산구,3
3,11,서울특별시,200,성동구,4
4,11,서울특별시,215,광진구,5


In [58]:
print("\n--- 관광정보 테이블에 시도명 및 시군구명 매칭 중 ---")

# 결합 키 문자열 변환 및 공백 제거
df_area_list_all['lDongRegnCd'] = df_area_list_all['lDongRegnCd'].astype(str).str.strip()
df_area_list_all['lDongSignguCd'] = df_area_list_all['lDongSignguCd'].astype(str).str.strip()
df_sido['lDongRegnCd'] = df_sido['lDongRegnCd'].astype(str).str.strip()
df_sido['lDongSignguCd'] = df_sido['lDongSignguCd'].astype(str).str.strip()

df_sido_clean = df_sido[['lDongRegnCd', 'lDongRegnNm', 'lDongSignguCd', 'lDongSignguNm']].drop_duplicates()

df_final = pd.merge(
    df_area_list_all,
    df_sido_clean,
    on=['lDongRegnCd', 'lDongSignguCd'],
    how='left'
)

print("매칭 완료! 데이터 크기:", df_final.shape)


--- 관광정보 테이블에 시도명 및 시군구명 매칭 중 ---
매칭 완료! 데이터 크기: (50677, 22)


In [59]:
print("\n--- 결합 실패(NaN) 건 및 주소 기반 지역명 자동 보정 ---")

nan_count = df_final['lDongRegnNm'].isna().sum()
print(f"정제 전 결합 실패(NaN) 개수: {nan_count}개")

def fill_missing_regions(row):
    sido = row['lDongRegnNm']
    sigungu = row['lDongSignguNm']
    
    # 1. 이미 정상 매칭 완료된 경우 그대로 반환
    if pd.notna(sido) and pd.notna(sigungu) and str(sido).strip() != '' and str(sigungu).strip() != '':
        return sido, sigungu
        
    addr = str(row.get('addr1') or '').strip()
    
    sido_map = {
        '서울': '서울특별시', '부산': '부산광역시', '대구': '대구광역시',
        '인천': '인천광역시', '광주': '광주광역시', '대전': '대전광역시',
        '울산': '울산광역시', '세종': '세종특별자치시', '경기': '경기도',
        '강원': '강원특별자치도', '충북': '충청북도', '충청북': '충청북도',
        '충남': '충청남도', '충청남': '충청남도', '전북': '전북특별자치도',
        '전라북': '전북특별자치도', '전남': '전라남도', '전라남': '전라남도',
        '경북': '경상북도', '경상북': '경상북도', '경남': '경상남도',
        '경상남': '경상남도', '제주': '제주특별자치도'
    }
    
    # 2. 주소(addr1) 정보가 있다면 주소 기반으로 최우선 파싱 (코드 정보 오류 대비)
    if addr:
        tokens = addr.split()
        if len(tokens) >= 1:
            first_token = tokens[0]
            parsed_sido = None

            for key, val in sido_map.items():
                if first_token.startswith(key):
                    parsed_sido = val
                    break
                
            if not parsed_sido:
                parsed_sido = first_token
                
            parsed_sigungu = None
            if len(tokens) >= 3 and tokens[1].endswith('시') and tokens[2].endswith('구'):
                parsed_sigungu = tokens[1] + ' ' + tokens[2]
            elif len(tokens) >= 2:
                parsed_sigungu = tokens[1]
            else:
                parsed_sigungu = parsed_sido
                
            return parsed_sido, parsed_sigungu

    # 3. 주소 정보가 없고, 시도 코드(lDongRegnCd)만 있을 경우 해당 시도 매칭 (시군구는 '미분류')
    reg_cd = str(row.get('lDongRegnCd') or '').strip()
    if reg_cd and reg_cd.lower() != 'nan':
        cd_map = {
            '11': '서울특별시', '26': '부산광역시', '27': '대구광역시', '28': '인천광역시',
            '29': '광주광역시', '30': '대전광역시', '31': '울산광역시', '36': '세종특별자치시',
            '41': '경기도', '42': '강원특별자치도', '43': '충청북도', '44': '충청남도',
            '45': '전북특별자치도', '46': '전라남도', '47': '경상북도', '48': '경상남도',
            '50': '제주특별자치도', '51': '강원특별자치도', '52': '전북특별자치도'
        }
        sido_val = cd_map.get(reg_cd, '미분류')
        return sido_val, '미분류'
        
    # 4. 주소와 시도 코드 둘 다 없으면 최종 결측치 처리
    return '미분류', '미분류'

parsed_regions = df_final.apply(fill_missing_regions, axis=1)
df_final['lDongRegnNm'] = [r[0] for r in parsed_regions]
df_final['lDongSignguNm'] = [r[1] for r in parsed_regions]

new_nan_count = df_final['lDongRegnNm'].isna().sum()
print(f"정제 후 결합 실패(NaN) 개수: {new_nan_count}개")


--- 결합 실패(NaN) 건 및 주소 기반 지역명 자동 보정 ---
정제 전 결합 실패(NaN) 개수: 31개
정제 후 결합 실패(NaN) 개수: 0개


In [60]:
print("\n--- 카테고리(대/중/소분류) 코드 수집 및 매칭 ---")

cat_dfs = []
page_no = 1
rows_per_page = 300

while True:
    cat_params = {
        "lclsSystmListYn": "Y",
        "numOfRows": rows_per_page,
        "pageNo": page_no
    }
    df_cat_chunk = fetch_api_data(endpoint="lclsSystmCode2", params=cat_params)

    if df_cat_chunk.empty:
        break
    cat_dfs.append(df_cat_chunk)

    if len(df_cat_chunk) < rows_per_page:
        break

    page_no += 1
    time.sleep(0.2)

df_categories = pd.concat(cat_dfs, ignore_index=True)
print("카테고리 코드 수집 완료! 총 개수:", df_categories.shape)

df_categories_clean = df_categories[[
    'lclsSystm1Cd', 'lclsSystm1Nm',
    'lclsSystm2Cd', 'lclsSystm2Nm',
    'lclsSystm3Cd', 'lclsSystm3Nm'
]].drop_duplicates()

df_categories_clean.rename(columns={
    'lclsSystm1Cd': 'lclsSystm1',
    'lclsSystm2Cd': 'lclsSystm2',
    'lclsSystm3Cd': 'lclsSystm3'
}, inplace=True)

df_categories_clean['lclsSystm1'] = df_categories_clean['lclsSystm1'].astype(str).str.strip()
df_categories_clean['lclsSystm2'] = df_categories_clean['lclsSystm2'].astype(str).str.strip()
df_categories_clean['lclsSystm3'] = df_categories_clean['lclsSystm3'].astype(str).str.strip()
df_final['lclsSystm1'] = df_final['lclsSystm1'].astype(str).str.strip()
df_final['lclsSystm2'] = df_final['lclsSystm2'].astype(str).str.strip()
df_final['lclsSystm3'] = df_final['lclsSystm3'].astype(str).str.strip()

df_final = pd.merge(
    df_final,
    df_categories_clean,
    on=['lclsSystm1', 'lclsSystm2', 'lclsSystm3'],
    how='left'
)

# 로컬 카테고리 매핑 정의서 보완(Fallback)
import json
json_path_def = '../data/신분류체계정보_관광타입정보_연계_정의서_exp.json'
if os.path.exists(json_path_def):
    try:
        with open(json_path_def, 'r', encoding='utf-8') as f:
            local_cat_map = json.load(f)
        
        missing_cat_mask = df_final['lclsSystm3Nm'].isna()
        print(f'카테고리 누락 건수 (보완 전): {missing_cat_mask.sum()}개')
        
        if missing_cat_mask.sum() > 0:
            for idx, row in df_final[missing_cat_mask].iterrows():
                code = str(row['lclsSystm3']).strip()
                if code in local_cat_map:
                    df_final.at[idx, 'lclsSystm1Nm'] = local_cat_map[code].get('대분류명')
                    df_final.at[idx, 'lclsSystm2Nm'] = local_cat_map[code].get('중분류명')
                    df_final.at[idx, 'lclsSystm3Nm'] = local_cat_map[code].get('소분류명')
    except Exception as e:
        print("로컬 카테고리 매핑 보완 중 에러:", e)

print("최종 카테고리 매칭 완료!")
cat_nan_count = df_final['lclsSystm3Nm'].isna().sum()
print(f"최종 카테고리 매칭 누락 건수: {cat_nan_count}개")


--- 카테고리(대/중/소분류) 코드 수집 및 매칭 ---
카테고리 코드 수집 완료! 총 개수: (245, 7)
카테고리 누락 건수 (보완 전): 4개
최종 카테고리 매칭 완료!
최종 카테고리 매칭 누락 건수: 0개


## 4. 최종 데이터 파일 정리 및 내보내기 (CSV 단일화)

In [61]:
print("\n--- 최종 데이터셋 정리 및 CSV 저장 ---")

# 1. 관광타입ID -> 한글 명칭 변환 매핑
content_type_map = {
    '12': '관광지',
    '14': '문화시설',
    '15': '축제/공연/행사',
    '25': '여행 코스',
    '28': '레포츠',
    '32': '숙박',
    '38': '쇼핑',
    '39': '음식점'
}
df_final['contenttypename'] = df_final['contenttypeid'].astype(str).str.strip().map(content_type_map)

# 2. 메인 데이터프레임 복사본 준비
df_main_export = df_final.copy()

# 3. 여행 코스 분리 및 제거
df_courses = df_main_export[df_main_export['contenttypename'] == '여행 코스'].copy()
df_main_export = df_main_export[df_main_export['contenttypename'] != '여행 코스'].copy()
print(f"여행 코스 분리 완료: {len(df_courses)}행 제외됨 (단일 POI 장소만 유지)")

# 4. 물리적 좌표(소수점 4자리 반올림) 및 유사 장소명 중복 제거
df_main_export['mapx_round'] = df_main_export['mapx'].astype(float).round(4)
df_main_export['mapy_round'] = df_main_export['mapy'].astype(float).round(4)
df_main_export['has_image'] = df_main_export['firstimage'].notna().astype(int)
df_main_export.sort_values(by=['mapx_round', 'mapy_round', 'title', 'has_image'], ascending=[True, True, True, False], inplace=True)

before_count = len(df_main_export)
df_main_export.drop_duplicates(subset=['mapx_round', 'mapy_round', 'title'], keep='first', inplace=True)
df_main_export.drop(columns=['mapx_round', 'mapy_round', 'has_image'], inplace=True, errors='ignore')
print(f"중복 장소 제거 완료: {before_count}행 -> {len(df_main_export)}행 (제거 건수: {before_count - len(df_main_export)}행)")

# 5. 불필요한 코드 컬럼들 제거
cols_to_drop = ['lDongRegnCd', 'lDongSignguCd', 'lclsSystm1', 'lclsSystm2', 'lclsSystm3', 'contenttypeid']
df_main_export.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# 6. 메인 컬럼 순서 재정렬 (title, contentid, contenttypename을 맨 앞으로)
front_cols = ['title', 'contentid', 'contenttypename']
remaining_cols = [col for col in df_main_export.columns if col not in front_cols]
df_main_export = df_main_export[front_cols + remaining_cols]

# 7. 파일 내보내기
path_main_csv = "../data/관광정보_메인_장소_데이터.csv"
path_course_csv = "../data/관광정보_여행코스_참조_데이터.csv"
os.makedirs("../data", exist_ok=True)
df_main_export.to_csv(path_main_csv, index=False, encoding='utf-8-sig')
df_courses.to_csv(path_course_csv, index=False, encoding='utf-8-sig')
print(f"[메인 데이터 저장 완료] {path_main_csv} ({len(df_main_export)}행)")
print(f"[여행코스 데이터 저장 완료] {path_course_csv} ({len(df_courses)}행)")

# 8. 기존 생성된 JSON 파일 삭제 (CSV 단일화 정책)
for file_name in ["관광정보_메인_장소_데이터.json", "관광정보_소개정보.json", "관광정보_세부반복정보.json", "관광정보_추가이미지.json"]:
    json_file_path = os.path.join("../data", file_name)
    if os.path.exists(json_file_path):
        try:
            os.remove(json_file_path)
            print(f"- 기존 JSON 파일 삭제 완료: {json_file_path}")
        except Exception as e:
            print(f"- {json_file_path} 삭제 실패: {e}")



--- 최종 데이터셋 정리 및 CSV 저장 ---
[메인 데이터 저장 완료] ../data/관광정보_메인_장소_데이터.csv (50677행)
